<h1><strong>1. Project overview</strong></h1>
<ol type="a">
  <li>Documentation is found in the Project documentation folder.</li>
  <li>This note-book is the data pipeline that enriches the provider collection in the Mongo db. repository </li>
</ol>


<hr style="border: none; height: 2px; background-color: blue; margin-top: 6px; margin-bottom: 0;" />
<small>Add all imports and get the DB</small>

In [ ]:
from __future__ import annotations

# --- Standard library ---
import csv
import math
import os
import re
from dataclasses import asdict, dataclass, field
from decimal import Decimal, ROUND_HALF_UP
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

# --- Third-party ---
import pandas as pd
import json
from bson.decimal128 import Decimal128
from dotenv import load_dotenv
from pymongo import MongoClient
from pymongo.collection import Collection

# --- Local / project ---
import sys
_code_dir = Path.cwd() if (Path.cwd() / "Enrichment").exists() else Path.cwd() / "code"
if _code_dir.exists() and str(_code_dir) not in sys.path:
    sys.path.insert(0, str(_code_dir))

from Utilities.ChatHealthyMongoUtilities import ChatHealthyMongoUtilities
from Enrichment.enrich_provider_records import EnrichProviderRecordsWithCountyData


load_dotenv()  # <-- REQUIRED for .env files

conn_str = os.getenv("MONGO_connectionString")
if not conn_str:
    raise EnvironmentError("MONGO_connectionString is not set")

DbUtil = ChatHealthyMongoUtilities(conn_str)
DB = DbUtil.getConnection()



In [ ]:
from IPython.display import HTML, display

# Wrap long output text in notebook cells

display(HTML("""
<style>
.output_area pre {
    white-space: pre-wrap !important;
    word-break: break-word !important;
}
</style>
"""))

Get zip -to county csv and load it to the db but also get the FIPS county name file so the file can be enriched with the county name cross walk. 

In [ ]:

# -----------------------------
# Results
# -----------------------------
@dataclass(frozen=True)
class LoadResults:
    # HUD source (rows from ZIP_COUNTY_*.csv)
    hud_rows_read: int
    hud_rows_skipped_missing_required: int  # missing ZIP/COUNTY/TOT_RATIO after parsing
    hud_rows_skipped_non_county_fips: int   # COUNTY not present in county-level FIPS map (Summary Level != 050)
    hud_rows_considered_for_normalization: int  # rows that passed validation and county-only filter
    hud_rows_written: int                  # number of normalized ZIP docs written (unique ZIPs)

    # County lookup source (Census all-geocodes)
    county_lookup_rows_read: int
    county_lookup_rows_written: int
    county_lookup_counties_in_map: int     # how many county FIPS were captured (Summary Level 050)

    # Raw HUD crosswalk (loaded AS-IS)
    census_crosswalk_rows_read: int
    census_crosswalk_rows_written: int

    # Enrichment metric
    normalized_zips_missing_county_name: int  # should be 0 when county-only filter is enforced

    # Validation metrics
    raw_hud_distinct_zips: int
    normalized_distinct_zips: int


# -----------------------------
# Helpers
# -----------------------------
def _is_collection_empty(col: Collection) -> bool:
    return col.estimated_document_count() == 0


def _s(v: Any) -> Optional[str]:
    if v is None:
        return None
    s = str(v).strip()
    return s if s != "" else None


def _zip5(value: Any) -> Optional[str]:
    if value is None:
        return None
    s = str(value).strip()
    if s == "":
        return None
    digits = "".join(ch for ch in s if ch.isdigit())
    if len(digits) == 0:
        return None
    if len(digits) > 5:
        digits = digits[:5]
    return digits.zfill(5)


def _to_decimal128(value: Any) -> Optional[Decimal128]:
    """
    Convert a numeric-looking value into BSON Decimal128.
    Returns None if the value is blank/unparseable.
    """
    if value is None:
        return None
    s = str(value).strip()
    if s == "":
        return None
    try:
        return Decimal128(Decimal(s))
    except (InvalidOperation, ValueError):
        return None


def _read_csv_rows(path: str) -> Tuple[List[str], List[Dict[str, Any]]]:
    with open(path, "r", newline="", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        if not reader.fieldnames:
            raise ValueError(f"No header row found in CSV: {path}")
        headers = list(reader.fieldnames)
        rows = [r for r in reader]
        return headers, rows


def _read_xlsx_rows(path: str) -> Tuple[List[str], List[Dict[str, Any]]]:
    from openpyxl import load_workbook  # type: ignore

    wb = load_workbook(path, read_only=True, data_only=True)
    ws = wb.active

    rows_iter = ws.iter_rows(values_only=True)
    try:
        header_row = next(rows_iter)
    except StopIteration:
        raise ValueError(f"Empty XLSX: {path}")

    headers = [str(h).strip() if h is not None else "" for h in header_row]
    if not any(headers):
        raise ValueError(f"Header row appears empty in XLSX: {path}")

    out: List[Dict[str, Any]] = []
    for row in rows_iter:
        doc: Dict[str, Any] = {}
        for i, h in enumerate(headers):
            if h == "":
                continue
            doc[h] = row[i] if i < len(row) else None
        if any(v is not None and str(v).strip() != "" for v in doc.values()):
            out.append(doc)

    return headers, out


def _read_tabular_rows(path: str) -> Tuple[List[str], List[Dict[str, Any]]]:
    ext = os.path.splitext(path.lower())[1]
    if ext in [".csv", ".txt"]:
        return _read_csv_rows(path)
    if ext in [".xlsx", ".xlsm"]:
        return _read_xlsx_rows(path)
    raise ValueError(f"Unsupported file type '{ext}'. Use .csv or .xlsx/.xlsm: {path}")


def _insert_many_in_batches(
    col: Collection,
    docs: List[Dict[str, Any]],
    batch_size: int = 1000,
) -> int:
    if not docs:
        return 0
    total = 0
    for i in range(0, len(docs), batch_size):
        chunk = docs[i : i + batch_size]
        col.insert_many(chunk, ordered=False)
        total += len(chunk)
    return total


def _normalize_key(s: str) -> str:
    return "".join(ch.lower() for ch in s if ch.isalnum())


def _find_col(headers: List[str], candidates: List[str]) -> Optional[str]:
    norm_map = {_normalize_key(h): h for h in headers if h is not None}
    for c in candidates:
        key = _normalize_key(c)
        if key in norm_map:
            return norm_map[key]
    return None


def _build_fips_to_county_name_map_from_rows(
    headers: List[str],
    rows: List[Dict[str, Any]],
) -> Dict[str, str]:
    """
    Builds map: 5-digit county FIPS -> county name.

    Supports your "all-geocodes-v2021.csv" format:
      - Summary Level
      - State Code (FIPS)
      - County Code (FIPS)
      - Area Name (including legal/statistical area description)

    County rows use Summary Level == '050'.
    countyFips = zfill2(State Code) + zfill3(County Code)

    Also includes fallback logic for other Census schemas.
    """
    summary_col = _find_col(headers, ["Summary Level"])
    state_code_col = _find_col(headers, ["State Code (FIPS)"])
    county_code_col = _find_col(headers, ["County Code (FIPS)"])
    area_name_col = _find_col(headers, ["Area Name (including legal/statistical area description)"])

    is_all_geocodes = all([summary_col, state_code_col, county_code_col, area_name_col])

    out: Dict[str, str] = {}

    if is_all_geocodes:
        for row in rows:
            summary = str(row.get(summary_col, "")).strip()
            if summary != "050":
                continue

            st_raw = row.get(state_code_col)
            co_raw = row.get(county_code_col)
            nm_raw = row.get(area_name_col)

            st = "".join(ch for ch in str(st_raw).strip() if ch.isdigit()) if st_raw is not None else ""
            co = "".join(ch for ch in str(co_raw).strip() if ch.isdigit()) if co_raw is not None else ""
            name = str(nm_raw).strip() if nm_raw is not None else ""

            if st == "" or co == "" or name == "":
                continue

            county_fips = st.zfill(2) + co.zfill(3)
            out.setdefault(county_fips, name)

        if not out:
            raise ValueError(
                "Detected an all-geocodes style file, but found zero county rows (Summary Level '050'). "
                "Inspect the 'Summary Level' column and tell me the value used for counties if it differs."
            )
        return out

    # Fallback generic schema
    statefp_col = _find_col(headers, ["STATEFP", "STATEFP20", "STATEFP10", "StateFP"])
    countyfp_col = _find_col(headers, ["COUNTYFP", "COUNTYFP20", "COUNTYFP10", "CountyFP"])
    fips_col = _find_col(headers, ["FIPS", "GEOID", "COUNTYFIPS", "COUNTY_FIPS", "COUNTYFP"])
    name_col = _find_col(headers, ["COUNTYNAME", "COUNTY_NAME", "NAMELSAD", "NAME"])

    if name_col is None:
        raise ValueError(
            "Could not find a county name column in the county lookup file. "
            f"Headers seen: {headers}"
        )

    for row in rows:
        nm_raw = row.get(name_col)
        county_name = str(nm_raw).strip() if nm_raw is not None else ""
        if county_name == "":
            continue

        fips: Optional[str] = None

        if statefp_col and countyfp_col:
            st = row.get(statefp_col)
            co = row.get(countyfp_col)
            if st is not None and co is not None:
                st_s = "".join(ch for ch in str(st) if ch.isdigit()).zfill(2)
                co_s = "".join(ch for ch in str(co) if ch.isdigit()).zfill(3)
                if len(st_s) == 2 and len(co_s) == 3:
                    fips = st_s + co_s

        if fips is None and fips_col:
            v = row.get(fips_col)
            if v is not None:
                digits = "".join(ch for ch in str(v) if ch.isdigit())
                if len(digits) >= 5:
                    fips = digits[-5:]

        if fips:
            out.setdefault(fips, county_name)

    if not out:
        raise ValueError(
            "Built an empty county FIPS -> county name map. Headers may not match expected patterns."
        )

    return out


# -----------------------------
# Main loader
# -----------------------------
def load_county_zip_with_census_collections(
    argDbConnection: MongoClient,
    argDatabaseName: str,

    # 2 source files
    argHudCountyZipCsvPath: str,
    argCountyFipsNameFilePath: str,

    # 3 collections
    argHudNormalizedCollectionName: str,
    argCountyLookupCollectionName: str,
    argCensusCrosswalkCollectionName: str,

    argBatchSize: int = 1000,
) -> LoadResults:
    """
    Creates/loads THREE collections, using TWO source files:

    Source file #1: argHudCountyZipCsvPath (HUD ZIP-County crosswalk CSV)
      - Loaded AS-IS into argCensusCrosswalkCollectionName
      - Also normalized into argHudNormalizedCollectionName (highest TOT_RATIO per ZIP)
      - Normalized collection is restricted to COUNTY FIPS that exist in the county-only
        lookup map (Summary Level 050).

    Source file #2: argCountyFipsNameFilePath (County FIPS -> County Name reference)
      - Loaded AS-IS into argCountyLookupCollectionName
      - Used to enrich normalized records with countyName
      - Supports Census "all-geocodes" CSV format

    Normalized fields written:
      ZIP                 -> zip (ZIP5 string)
      COUNTY              -> countyFips (5-digit string)
      (derived)           -> countyName (string, from Summary Level 050 rows only)
      USPS_ZIP_PREF_CITY  -> mainCity
      USPS_ZIP_PREF_STATE -> state
      TOT_RATIO           -> percentOfZipInCounty (BSON Decimal128)

    Loads each collection only if empty, and returns a detailed validation report.
    """
    db = argDbConnection[argDatabaseName]

    normalized_col = db[argHudNormalizedCollectionName]
    county_lookup_col = db[argCountyLookupCollectionName]
    census_crosswalk_col = db[argCensusCrosswalkCollectionName]

    # Read HUD crosswalk once
    hud_headers, hud_rows = _read_csv_rows(argHudCountyZipCsvPath)

    required = ["ZIP", "COUNTY", "USPS_ZIP_PREF_CITY", "USPS_ZIP_PREF_STATE", "TOT_RATIO"]
    missing = [h for h in required if h not in hud_headers]
    if missing:
        raise ValueError(
            "HUD CSV is missing required columns: "
            f"{missing}. Headers seen: {hud_headers}"
        )

    # Compute raw distinct zips in HUD (for validation)
    raw_zip_set = set()
    for row in hud_rows:
        z = _zip5(row.get("ZIP"))
        if z:
            raw_zip_set.add(z)
    raw_hud_distinct_zips = len(raw_zip_set)

    # 1) Load HUD crosswalk AS-IS into raw collection
    census_crosswalk_rows_read = len(hud_rows)
    census_crosswalk_rows_written = 0
    if _is_collection_empty(census_crosswalk_col):
        census_crosswalk_rows_written = _insert_many_in_batches(
            census_crosswalk_col, hud_rows, batch_size=argBatchSize
        )

    # 2) Load county lookup AS-IS into lookup collection
    county_headers, county_rows = _read_tabular_rows(argCountyFipsNameFilePath)

    county_lookup_rows_read = len(county_rows)
    county_lookup_rows_written = 0
    if _is_collection_empty(county_lookup_col):
        county_lookup_rows_written = _insert_many_in_batches(
            county_lookup_col, county_rows, batch_size=argBatchSize
        )

    # Build county-only map (Summary Level 050)
    fips_to_name = _build_fips_to_county_name_map_from_rows(county_headers, county_rows)
    county_lookup_counties_in_map = len(fips_to_name)

    # 3) Normalize HUD + enrich with countyName; enforce county-only via fips_to_name membership
    hud_rows_read = 0
    hud_rows_skipped_missing_required = 0
    hud_rows_skipped_non_county_fips = 0
    hud_rows_considered_for_normalization = 0

    hud_rows_written = 0
    normalized_zips_missing_county_name = 0

    if _is_collection_empty(normalized_col):
        best_by_zip: Dict[str, Dict[str, Any]] = {}

        for row in hud_rows:
            hud_rows_read += 1

            zip5 = _zip5(row.get("ZIP"))
            county_digits = "".join(ch for ch in str(row.get("COUNTY", "")).strip() if ch.isdigit())
            county_fips = county_digits.zfill(5) if county_digits else None
            ratio128 = _to_decimal128(row.get("TOT_RATIO"))

            if zip5 is None or county_fips is None or ratio128 is None:
                hud_rows_skipped_missing_required += 1
                continue

            # COUNTY-ONLY FILTER: Only accept counties present in the 050-only map
            county_name = fips_to_name.get(county_fips)
            if county_name is None:
                hud_rows_skipped_non_county_fips += 1
                continue

            hud_rows_considered_for_normalization += 1

            candidate = {
                "zip": zip5,
                "countyFips": county_fips,
                "countyName": county_name,  # should always be present due to filter
                "mainCity": _s(row.get("USPS_ZIP_PREF_CITY")),
                "state": _s(row.get("USPS_ZIP_PREF_STATE")),
                "percentOfZipInCounty": ratio128,  # Decimal128
            }

            current = best_by_zip.get(zip5)
            if current is None:
                best_by_zip[zip5] = candidate
            else:
                cur_val = current["percentOfZipInCounty"].to_decimal()
                new_val = ratio128.to_decimal()
                if new_val > cur_val:
                    best_by_zip[zip5] = candidate

        docs = list(best_by_zip.values())

        # Should be zero now, but keep metric anyway
        normalized_zips_missing_county_name = sum(1 for d in docs if not d.get("countyName"))

        hud_rows_written = _insert_many_in_batches(
            normalized_col, docs, batch_size=argBatchSize
        )

        # Helpful indexes for your common queries
        normalized_col.create_index("zip")
        normalized_col.create_index([("zip", 1), ("countyFips", 1)])
        county_lookup_col.create_index("Summary Level")
        county_lookup_col.create_index("State Code (FIPS)")
        county_lookup_col.create_index("County Code (FIPS)")
        census_crosswalk_col.create_index("ZIP")
        census_crosswalk_col.create_index("COUNTY")

    else:
        # If collection already exists, we still return validation stats from sources,
        # but we will not rewrite normalized data.
        hud_rows_read = len(hud_rows)
        # We won't recompute these without reading DB; leave them as 0 to avoid false claims.
        hud_rows_skipped_missing_required = 0
        hud_rows_skipped_non_county_fips = 0
        hud_rows_considered_for_normalization = 0
        hud_rows_written = 0
        normalized_zips_missing_county_name = 0

    # Validation: distinct zips in normalized collection
    normalized_distinct_zips = normalized_col.distinct("zip")
    normalized_distinct_zips_count = len(normalized_distinct_zips)

    return LoadResults(
        hud_rows_read=hud_rows_read,
        hud_rows_skipped_missing_required=hud_rows_skipped_missing_required,
        hud_rows_skipped_non_county_fips=hud_rows_skipped_non_county_fips,
        hud_rows_considered_for_normalization=hud_rows_considered_for_normalization,
        hud_rows_written=hud_rows_written,
        county_lookup_rows_read=county_lookup_rows_read,
        county_lookup_rows_written=county_lookup_rows_written,
        county_lookup_counties_in_map=county_lookup_counties_in_map,
        census_crosswalk_rows_read=census_crosswalk_rows_read,
        census_crosswalk_rows_written=census_crosswalk_rows_written,
        normalized_zips_missing_county_name=normalized_zips_missing_county_name,
        raw_hud_distinct_zips=raw_hud_distinct_zips,
        normalized_distinct_zips=normalized_distinct_zips_count,
    )



Enrich countyzip with GDP data

In [ ]:
# File: EnrichCountyZipWithCountyGDPData.ipynb (single cell)
# Author: Skip Snow
# Co-Author: GPT-5
# Copyright (c) 2025 Skip Snow. All rights reserved.

from __future__ import annotations

# -----------------------------
# Imports
# -----------------------------
import re
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
from pymongo import MongoClient
from pymongo.collection import Collection

# -----------------------------
# Constants
# -----------------------------
_US_STATE_ABBR_TO_NAME: Dict[str, str] = {
    "AL": "Alabama","AK": "Alaska","AZ": "Arizona","AR": "Arkansas","CA": "California",
    "CO": "Colorado","CT": "Connecticut","DE": "Delaware","DC": "District of Columbia",
    "FL": "Florida","GA": "Georgia","HI": "Hawaii","ID": "Idaho","IL": "Illinois",
    "IN": "Indiana","IA": "Iowa","KS": "Kansas","KY": "Kentucky","LA": "Louisiana",
    "ME": "Maine","MD": "Maryland","MA": "Massachusetts","MI": "Michigan","MN": "Minnesota",
    "MS": "Mississippi","MO": "Missouri","MT": "Montana","NE": "Nebraska","NV": "Nevada",
    "NH": "New Hampshire","NJ": "New Jersey","NM": "New Mexico","NY": "New York",
    "NC": "North Carolina","ND": "North Dakota","OH": "Ohio","OK": "Oklahoma",
    "OR": "Oregon","PA": "Pennsylvania","RI": "Rhode Island","SC": "South Carolina",
    "SD": "South Dakota","TN": "Tennessee","TX": "Texas","UT": "Utah","VT": "Vermont",
    "VA": "Virginia","WA": "Washington","WV": "West Virginia","WI": "Wisconsin","WY": "Wyoming"
}
_US_STATE_NAMES = set(_US_STATE_ABBR_TO_NAME.values())
_TERRITORY_ABBRS = {"PR", "GU", "VI", "AS", "MP"}

_SUFFIX_PATTERNS = [
    r"\bcounty\b", r"\bparish\b", r"\bborough\b", r"\bcensus\s+area\b",
    r"\bmunicipality\b", r"\bmunicipio\b", r"\bcity\s+and\s+borough\b", r"\bcity\b",
]

# -----------------------------
# Dataclasses
# -----------------------------
@dataclass
class GDPRow:
    state_name: str
    county_norm: str
    real_gdp_2023_dollars: Optional[int]
    gdp_rank_in_state_2023: Optional[int]
    pct_change_2023: Optional[float]
    pct_change_rank_in_state_2023: Optional[int]


# -----------------------------
# Helpers
# -----------------------------
def _is_nan(v: Any) -> bool:
    try:
        return v is None or pd.isna(v)
    except Exception:
        return v is None

def _to_str(v: Any) -> Optional[str]:
    if _is_nan(v):
        return None
    s = str(v).strip()
    return s if s else None

def _to_float_safe(v: Any) -> Optional[float]:
    try:
        if _is_nan(v):
            return None
        if isinstance(v, str):
            s = v.strip()
            if s in {"", "--", "(D)", "(NA)"}:
                return None
            return float(s.replace(",", ""))
        return float(v)
    except Exception:
        return None

def _to_int_safe(v: Any) -> Optional[int]:
    f = _to_float_safe(v)
    if f is None:
        return None
    try:
        return int(f)
    except Exception:
        return None

def _normalize_county_name(name: str) -> str:
    if not name:
        return ""
    s = str(name).strip().lower()

    # Drop anything after a comma
    if "," in s:
        s = s.split(",", 1)[0].strip()

    # "County of X"
    s = re.sub(r"^\s*county\s+of\s+", "", s)

    # punctuation normalization
    s = s.replace("&", " and ")
    s = s.replace("-", " ")
    s = s.replace(".", "")
    s = s.replace("’", "'")
    s = re.sub(r"\bst\b", "saint", s)

    # strip suffix types
    for pat in _SUFFIX_PATTERNS:
        s = re.sub(pat, " ", s)

    s = re.sub(r"\s+", " ", s).strip()
    return s

def _state_abbr_to_name(abbr: str) -> Optional[str]:
    return _US_STATE_ABBR_TO_NAME.get(abbr)

def _is_us_total(name: str) -> bool:
    return name.strip().lower() == "united states"

def _looks_like_state_row(name: str, rank_cell: Any) -> bool:
    """
    In this workbook format:
      - State rows have name == state full name AND rank in state is '--' (or blank)
      - County rows have an integer rank in state (even if county name equals a state name, e.g. "Iowa" county)
    """
    if name not in _US_STATE_NAMES:
        return False
    s = _to_str(rank_cell)
    if s is None:
        return True
    return s.strip() == "--"

# -----------------------------
# Excel Parsing (fixed-column BEA layout)
# -----------------------------
def _parse_county_gdp_excel(
    path: str
) -> Tuple[Dict[Tuple[str, str], GDPRow], int, int, int]:
    """
    Parses CountyGDP_12_2024.xlsx using fixed columns observed in the file:

    Col 0: GeoName (United States / State / County)
    Col 1: 2023 Real GDP (thousands of chained dollars)
    Col 5: 2023 rank in state (for GDP)  (state rows show '--')
    Col 6: 2023 percent change
    Col 9: 2023 rank in state (for percent change) (state rows show '--')

    Returns:
      gdp_map[(state_name, county_norm)] -> GDPRow
      spreadsheet_total_records
      spreadsheet_included_records (county rows ingested)
      spreadsheet_ignored_out_of_scope_records (headers/blanks/US/state rows/non-county)
    """
    df = pd.read_excel(path, header=None, engine="openpyxl")
    rows = df.values.tolist()

    gdp_map: Dict[Tuple[str, str], GDPRow] = {}

    total_records = 0
    included = 0
    ignored = 0

    current_state: Optional[str] = None

    for r in rows:
        # consider "records" as rows with a non-empty GeoName cell
        name_raw = _to_str(r[0] if len(r) > 0 else None)
        if not name_raw:
            continue

        total_records += 1
        name = name_raw.strip()

        # Ignore top header/title lines (they are not US/State/County)
        # We'll treat them as out-of-scope unless they become state context.
        if _is_us_total(name):
            current_state = None
            ignored += 1
            continue

        # Identify state row
        rank_cell = r[5] if len(r) > 5 else None
        if _looks_like_state_row(name, rank_cell):
            current_state = name
            ignored += 1
            continue

        # County rows require a current_state
        if not current_state:
            ignored += 1
            continue

        # Pull fields by fixed indices
        gdp_thousands = _to_float_safe(r[1] if len(r) > 1 else None)
        gdp_rank = _to_int_safe(r[5] if len(r) > 5 else None)
        pct_change = _to_float_safe(r[6] if len(r) > 6 else None)
        pct_rank = _to_int_safe(r[9] if len(r) > 9 else None)

        # If GDP is missing, it's not a county record we can use
        if gdp_thousands is None:
            ignored += 1
            continue

        # Multiply thousands -> dollars (as requested)
        real_gdp_2023 = int(round(gdp_thousands * 1000.0))

        county_norm = _normalize_county_name(name)

        g = GDPRow(
            state_name=current_state,
            county_norm=county_norm,
            real_gdp_2023_dollars=real_gdp_2023,
            gdp_rank_in_state_2023=gdp_rank,
            pct_change_2023=pct_change,
            pct_change_rank_in_state_2023=pct_rank,
        )
        gdp_map[(current_state, county_norm)] = g
        included += 1

    ignored = max(0, total_records - included - sum(1 for _ in []))  # keep simple, ignored is computed below

    # Better: compute ignored as total_records - included - (count of state/us rows we treated as ignored already)
    # We already incremented ignored for those; so just return ignored as tracked.
    # But we also incremented ignored for rows without current_state, etc. so it's accurate.
    return gdp_map, total_records, included, ignored

# -----------------------------
# Main ETL Function
# -----------------------------
def EnrichCountyZipWithCountyGDPData(
    argDbConnection: MongoClient,
    argDBName: str = "PublicHealthData",
    argSourceExcelFilePath: str = "/mnt/data/CountyGDP_12_2024.xlsx",
    argSourceCollectionName: str = "CountyZipNormalized",
    argDestinationCollectionName: str = "CountyZipEnriched",
) -> List[Any]:
    """
    Enriches CountyZipNormalized into CountyZipEnriched with 2023 county GDP measures.

    - Skips Mongo source records where countyName is null (not inserted into destination)
    - For territories (PR, GU, VI, AS, MP): inserts record with GDP fields = null
    - For states: matches by (state full name, normalized county name)
    - Drops destination collection before writing (drains it)
    - Returns list of stats dictionaries as requested
    """

    gdp_map, spreadsheet_total, spreadsheet_included, spreadsheet_ignored = _parse_county_gdp_excel(argSourceExcelFilePath)

    db = argDbConnection[argDBName]
    src: Collection = db[argSourceCollectionName]
    dst: Collection = db[argDestinationCollectionName]

    # Drain destination collection if it exists
    dst.drop()

    mongo_source_total = src.count_documents({})

    mongo_ignored_out_of_scope = 0  # countyName null
    mongo_ignored_unmatched = 0
    mongo_written = 0

    # Use sets to avoid enormous duplicate lists in unmatched output
    unmatched_sets: Dict[str, set] = {}

    batch: List[Dict[str, Any]] = []
    BATCH_SIZE = 5000

    cursor = src.find({}, no_cursor_timeout=True)
    try:
        for doc in cursor:
            county_name = doc.get("countyName")
            if not county_name:
                mongo_ignored_out_of_scope += 1
                continue

            state_abbr = doc.get("state")
            doc.pop("_id", None)

            # Territories: keep record, GDP fields null
            if state_abbr in _TERRITORY_ABBRS:
                doc.update({
                    "realGDP2023": None,
                    "gdpRankInState2023": None,
                    "percentChange2023": None,
                    "percentChangeRankInState2023": None,
                })
            else:
                state_name = _state_abbr_to_name(state_abbr) if state_abbr else None
                county_norm = _normalize_county_name(county_name)

                g = gdp_map.get((state_name, county_norm)) if state_name else None

                if g is None:
                    mongo_ignored_unmatched += 1
                    key = state_name or "UNKNOWN"
                    if key not in unmatched_sets:
                        unmatched_sets[key] = set()
                    unmatched_sets[key].add(str(county_name))

                    doc.update({
                        "realGDP2023": None,
                        "gdpRankInState2023": None,
                        "percentChange2023": None,
                        "percentChangeRankInState2023": None,
                    })
                else:
                    doc.update({
                        "realGDP2023": g.real_gdp_2023_dollars,
                        "gdpRankInState2023": g.gdp_rank_in_state_2023,
                        "percentChange2023": g.pct_change_2023,
                        "percentChangeRankInState2023": g.pct_change_rank_in_state_2023,
                    })

            batch.append(doc)
            mongo_written += 1

            if len(batch) >= BATCH_SIZE:
                dst.insert_many(batch, ordered=False)
                batch.clear()

        if batch:
            dst.insert_many(batch, ordered=False)
            batch.clear()

    finally:
        try:
            cursor.close()
        except Exception:
            pass

    unmatched_counties_by_state: Dict[str, List[str]] = {
        state: sorted(list(names)) for state, names in unmatched_sets.items()
    }

    return [
        {"spreadsheet_ignored_out_of_scope_records": spreadsheet_ignored},
        {"spreadsheet_included_records": spreadsheet_included},
        {"spreadsheet_total_records": spreadsheet_total},
        {"mongo_source_total_records": mongo_source_total},
        {"mongo_records_ignored_out_of_scope": mongo_ignored_out_of_scope},
        {"mongo_records_ignored_unmatched_county_name": mongo_ignored_unmatched},
        {"mongo_records_written_to_destination": mongo_written},
        {"unmatched_counties_by_state": unmatched_counties_by_state},
        {"run_stats_full": {
            "spreadsheet_total_records": spreadsheet_total,
            "spreadsheet_included_records": spreadsheet_included,
            "spreadsheet_ignored_out_of_scope_records": spreadsheet_ignored,
            "mongo_source_total_records": mongo_source_total,
            "mongo_records_ignored_out_of_scope": mongo_ignored_out_of_scope,
            "mongo_records_ignored_unmatched_county_name": mongo_ignored_unmatched,
            "mongo_records_written_to_destination": mongo_written,
            "unmatched_counties_by_state": unmatched_counties_by_state,
        }},
    ]



Load test questions

In [ ]:
# --- Standard library ---
import json

# --- Third-party ---
from pymongo import MongoClient

def load_TestQuestions_to_mongodb(
    client: MongoClient,
    database_name: str,
    collection_name: str,
    file_path: str,
    batch_size: int = 1000
) -> dict:
    """
    Load data from a JSONL file into a MongoDB collection.
    Clears the collection before inserting new data.
    
    Parameters:
    -----------
    client : MongoClient
        An active MongoDB client connection
    database_name : str
        Name of the database to use
    collection_name : str
        Name of the collection to insert data into (will be cleared first)
    file_path : str
        Path to the JSONL file to read
    batch_size : int, optional
        Number of documents to insert at once (default: 1000)
    
    Returns:
    --------
    dict
        A dictionary containing:
        - 'inserted_count': number of documents inserted
        - 'deleted_count': number of documents deleted before insert
        - 'status': 'success' or 'error'
        - 'message': status message
    
    Example:
    --------
    >>> from pymongo import MongoClient
    >>> client = MongoClient('mongodb://localhost:27017/')
    >>> result = load_jsonl_to_mongodb(
    ...     client, 
    ...     'my_database', 
    ...     'my_collection', 
    ...     'data.jsonl'
    ... )
    >>> print(result)
    """
    try:
        # Access the database and collection
        db = client[database_name]
        collection = db[collection_name]
        
        # Clear the collection (delete all existing documents)
        delete_result = collection.delete_many({})
        deleted_count = delete_result.deleted_count
        print(f"Cleared {deleted_count} existing documents from collection '{collection_name}'")
        
        # Read and parse JSONL file
        documents = []
        inserted_count = 0
        skipped_lines = 0
        print(f"Reading: {file_path}")
        with open(file_path, 'r', encoding='utf-8') as file:
            for line_num, line in enumerate(file, 1):
                stripped = line.strip()
                if not stripped:
                    continue
                try:
                    doc = json.loads(stripped)
                    documents.append(doc)

                    # Insert in batches for better performance
                    if len(documents) >= batch_size:
                        result = collection.insert_many(documents)
                        inserted_count += len(result.inserted_ids)
                        documents = []

                except json.JSONDecodeError as e:
                    skipped_lines += 1
                    print(f"Warning: Skipping invalid JSON at line {line_num}: {e}")

                if line_num % 500 == 0:
                    print(f"  Processed {line_num} lines, inserted {inserted_count} so far...")
        
        # Insert any remaining documents
        if documents:
            result = collection.insert_many(documents)
            inserted_count += len(result.inserted_ids)
        
        return {
            'status': 'success',
            'deleted_count': deleted_count,
            'inserted_count': inserted_count,
            'skipped_lines': skipped_lines,
            'message': f'Inserted {inserted_count} documents from {file_path}'
        }
        
    except FileNotFoundError:
        return {
            'status': 'error',
            'deleted_count': 0,
            'inserted_count': 0,
            'message': f'File not found: {file_path}'
        }
    except Exception as e:
        return {
            'status': 'error',
            'deleted_count': 0,
            'inserted_count': 0,
            'message': f'Error: {str(e)}'
        }

Driver

In [5]:
zipCountyCSVPath=r"C:\chatHealthy\Resources\ZIP_COUNTY_092025.csv"
CountyFipsNameFilePath=r"C:\chatHealthy\Resources\all-geocodes-v2021.csv"
GDP_FILE_PATH=r"C:\chatHealthy\Resources\CountyGDP_12_2024.xlsx" 


'''
stats = load_county_zip_with_census_collections(
    argDbConnection=DbUtil.getConnection(),
    argDatabaseName="PublicHealthData",
    argHudCountyZipCsvPath=zipCountyCSVPath,
    argCountyFipsNameFilePath=CountyFipsNameFilePath,
    argHudNormalizedCollectionName="CountyZipNormalized",
    argCountyLookupCollectionName="CountyFipsLookup",
    argCensusCrosswalkCollectionName="HudZipCountyRaw",
)

stats = EnrichCountyZipWithCountyGDPData(
    argDbConnection=DbUtil.getConnection(),
    argDBName="PublicHealthData",
    argSourceExcelFilePath=GDP_FILE_PATH,
    argSourceCollectionName="CountyZipNormalized",
    argDestinationCollectionName="CountyZipEnriched",
)

stats = load_TestQuestions_to_mongodb(
    DbUtil.getConnection(),
    "FindCareTest",
    "HowManyDocs",
    r"C:\chatHealthy\Resources\TestData\physicians_phd_QuantityTestQuestionsAndAnswers.jsonl",
    1000)
for k, v in stats.items():
    print(f"{k}: {v}")

# Verify: count documents and show where to find them
conn = DbUtil.getConnection()
db_name, coll_name = "FindCareTest", "HowManyDocs"
coll = conn[db_name][coll_name]
verify_count = coll.count_documents({})
print(f"\n--- Verification ---")
print(f"Database.Collection: {db_name}.{coll_name}")
print(f"Documents in collection: {verify_count}")
if verify_count > 0:
    sample = coll.find_one()
    print(f"Sample document keys: {list(sample.keys())[:8]}...")
print(f"Use MONGO_connectionString from .env in MongoDB Compass → {db_name} → {coll_name}")

'''
providerEnrichment = EnrichProviderRecordsWithCountyData(argMongoDataBaseClient=DbUtil.getConnection())
providerEnrichment.ProcessAllRecords()

2026-02-22 20:13:52,659  INFO      Enrichment.enrich_provider_records  Cleared existing 'EnrichedProvider' collection – 6975936 document(s) removed.
2026-02-22 20:13:53,270  INFO      Enrichment.enrich_provider_records  Initialised – 39063 unique ZIP codes loaded. First ZIP: 00501
2026-02-22 20:13:54,331  INFO      Enrichment.enrich_provider_records  Flushed batch of 500 – total inserted so far: 500
2026-02-22 20:13:54,686  INFO      Enrichment.enrich_provider_records  Flushed batch of 500 – total inserted so far: 1000
2026-02-22 20:13:55,036  INFO      Enrichment.enrich_provider_records  Flushed batch of 500 – total inserted so far: 1500
2026-02-22 20:13:55,457  INFO      Enrichment.enrich_provider_records  Flushed batch of 500 – total inserted so far: 2000
2026-02-22 20:13:55,847  INFO      Enrichment.enrich_provider_records  Flushed batch of 500 – total inserted so far: 2500
2026-02-22 20:13:56,238  INFO      Enrichment.enrich_provider_records  Flushed batch of 500 – total inserted 

True